# Step 2: Base Model Inference

Use **HuggingFaceTB/SmolLM2-1.7B-Instruct** to generate outputs for eval/test questions, serving as a baseline for comparison with the fine-tuned model.

**Workflow**:
1. Load the base model and tokenizer (the first run will download from HuggingFace)
2. Read `eval.jsonl` and `test.jsonl`
3. For each question, use the SmolLM2 Chat Template to construct a prompt and generate output
4. Write results to `outputs/base_outputs.jsonl` (including question, model_output, and ground_truth)

**Before running**: Run from the project root directory, or make sure the paths are correct.

## 1. Path Configuration

Set the project root directory and ensure the `data/` and `outputs/` paths are correct.

In [ ]:
from pathlib import Path

# Project root directory: if running under scripts/finetune, go up two levels
project_root = Path.cwd()
if project_root.name == "finetune":
    project_root = project_root.parent.parent
elif project_root.name == "scripts":
    project_root = project_root.parent

data_dir = project_root / "data"
output_dir = project_root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

eval_path = data_dir / "eval.jsonl"
test_path = data_dir / "test.jsonl"
output_path = output_dir / "base_outputs.jsonl"

print(f"Project root: {project_root}")
print(f"Eval: {eval_path} (exists: {eval_path.exists()})")
print(f"Test: {test_path} (exists: {test_path.exists()})")
print(f"Output: {output_path}")

Project root: /Users/wuyusen/Documents/bet
Eval: /Users/wuyusen/Documents/bet/data/eval.jsonl (exists: True)
Test: /Users/wuyusen/Documents/bet/data/test.jsonl (exists: True)
Output: /Users/wuyusen/Documents/bet/outputs/base_outputs.jsonl


## 2. Load Base Model and Tokenizer
On first run, this will download `HuggingFaceTB/SmolLM2-1.7B-Instruct` (~3.4GB) from the HuggingFace Hub.

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print(f"Loading model: {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

if not torch.cuda.is_available():
    model = model.to("cpu")

print(f"Model loaded. Device: {next(model.parameters()).device}")

/Users/wuyusen/Documents/bet/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer: HuggingFaceTB/SmolLM2-1.7B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model: HuggingFaceTB/SmolLM2-1.7B-Instruct


Loading weights: 100%|██████████| 218/218 [00:05<00:00, 40.05it/s, Materializing param=model.norm.weight]                              


Model loaded. Device: cpu


## 3. Load Evaluation Data
Combine eval and test sets. Assignment requires at least 10–15 questions. Here we use the full eval + test sets, about 199 questions in total.

In [5]:
import json

def load_jsonl(path: Path) -> list[dict]:
    """Read a JSONL file, one JSON object per line."""
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                items.append(json.loads(line))
    return items

eval_data = load_jsonl(eval_path) if eval_path.exists() else []
test_data = load_jsonl(test_path) if test_path.exists() else []

questions = eval_data + test_data

# Quick test: set to 15 to only run 15 questions (assignment requires at least 10-15 questions)
NUM_QUESTIONS = None  # None = all, or set to a number like 15
if NUM_QUESTIONS is not None:
    questions = questions[:NUM_QUESTIONS]
    print(f"(Limited to the first {NUM_QUESTIONS} questions)")

print(f"Eval: {len(eval_data)} records")
print(f"Test: {len(test_data)} records")
print(f"Total: {len(questions)} records")

if questions:
    print("\nExample (first record):")
    q = questions[0]
    print(f"  instruction: {q['instruction'][:80]}...")
    print(f"  input: {len(q['input'])} chars")
    print(f"  output: {len(q['output'])} chars")

Eval: 99 records
Test: 100 records
Total: 199 records

Example (first record):
  instruction: Should I bet Hunter Tyson over/under 2.5 points?...
  input: 2845 chars
  output: 1907 chars


## 4. Build Prompt (SmolLM2 Chat Template)
Convert Alpaca format (instruction, input) into SmolLM2's ChatML format using `apply_chat_template`.

In [6]:
SYSTEM_PROMPT = """You are an expert NBA player data analyst. Your task is to analyze betting questions (over/under) using historical player statistics.

## Tree of Thought Reasoning Framework

Build your reasoning as a tree with these main branches (evaluate each, then synthesize):

1. **Sample & Statistics Branch**: For each context filter (all games, last N games, starter vs bench, with/without star teammates), assess:
   - n_games: Is sample size sufficient? (n < 10 → low weight)
   - p_over, p_under, mean, std: Which context favors over vs under?
   - Conflict between contexts → note uncertainty

2. **Lineup/Teammate Branch**: How does starter vs bench, or with/without star teammates, change the stats? Does lineup context support or contradict the main trend?

3. **Risk & Synthesis Branch**: Given the above, what is the net signal? If branches conflict or sample is weak → prefer "avoid".

## Output Format (JSON only, no markdown)

{
  "decision": "over" | "under" | "avoid",
  "confidence": 0.0 to 1.0,
  "reasoning": {
    "tree_of_thought": [
      {"step": 1, "branch": "sample_stats", "thought": "...", "conclusion": "..."},
      {"step": 2, "branch": "lineup_teammate", "thought": "...", "conclusion": "..."},
      {"step": 3, "branch": "synthesis", "thought": "...", "conclusion": "..."}
    ]
  },
  "summary": "One-sentence conclusion"
}

Each step must have: branch (which dimension), thought (your analysis), conclusion (what this branch implies for the decision).
Respond with ONLY valid JSON, no markdown or extra text."""

def build_prompt(instruction: str, input_str: str) -> str:
    """
    Combine the instruction and input into a user message, and apply the SmolLM2 chat template.
    add_generation_prompt=True will append <|im_start|>assistant\n at the end, prompting the model to begin generating.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{instruction}\n\nStatistics:\n{input_str}"},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    return text

# Run sample prompt check
sample = build_prompt(questions[0]["instruction"], questions[0]["input"])
print("Sample prompt (first 500 chars):")
print(sample[:500])
print("...")

Sample prompt (first 500 chars):
<|im_start|>system
You are an expert NBA player data analyst. Your task is to analyze betting questions (over/under) using historical player statistics.

## Tree of Thought Reasoning Framework

Build your reasoning as a tree with these main branches (evaluate each, then synthesize):

1. **Sample & Statistics Branch**: For each context filter (all games, last N games, starter vs bench, with/without star teammates), assess:
   - n_games: Is sample size sufficient? (n < 10 → low weight)
   - p_over
...


## 5. Inference Loop

Generate output for each question. `max_new_tokens=4096` is sufficient for JSON output; you can adjust this if the input is particularly long.

In [ ]:
def generate(prompt: str) -> str:
    """Generate model output for the given prompt."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=4096,
        )
    
    # Only get the newly generated part (remove prompt)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print("Starting inference...")
results = []
for i, item in enumerate(questions):
    prompt = build_prompt(item["instruction"], item["input"])
    model_output = generate(prompt)
    results.append({
        "question": item["instruction"],
        "input": item["input"],
        "model_output": model_output.strip(),
        "ground_truth": item["output"],
    })
    if (i + 1) % 10 == 0:
        print(f"  Processed {i + 1}/{len(questions)} questions")

print(f"Completed. Total: {len(results)} questions")

Starting inference...
  Processed 10/199 questions
  Processed 20/199 questions


## 6. Write Results
Write the results to `outputs/base_outputs.jsonl` for use by subsequent compare_outputs.

In [ ]:
with open(output_path, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Written to: {output_path}")
print(f"Total: {len(results)} items")
print("\nExample (first 300 characters of first model_output):")
print(results[0]["model_output"][:300])